# D1.9 · Detections whose subject is the agent platform

**Function D — The Agentic SOC → The Agentic SOC — Detection**  ·  *AI for Security*

Builds on **[D1.8 · Threat intel sub-lane](https://spbreed.github.io/cyber-commons/lessons/D1.8.html)**.

| | |
|---|---|
| Tools used | Falco, Gitleaks, Sigstore |

## What this lesson is

**What it covers.** Run four platform detectors over one day of events and see which of them a generic anomaly score would have missed.

**Why a security engineer needs it.** Platform-layer compromise is invisible to workload-layer detection. The escape, the poisoned cache entry and the silently expired exemption all look like normal operation from inside. The control it builds is: named escape primitives rather than anomaly scoring (C1.4), cache integrity diffing against a manifest (C5.4), upload scanning (C3.4), secret scanning wired to automated revocation (C4.1), and exemption-state reconciliation (C6.3).

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

The escape, the poisoned cache entry and the silently expired exemption all look like normal operation from inside the workload. These are detections whose subject is the platform running the agent, and not one of them reads the agent's own logs.

> **At CyberTravels.** Detections whose subject is the platform CyberTravels runs on, not the agents themselves — the escape, the poisoned package cache, the credential loose on the internet, the guardrail still switched off after the demo.

## 2 · The framework

```
   subject of the detection = the platform, not the workload

   escape primitives      ptrace non-child . LD_PRELOAD . /proc/self/mem
                          unshare/setns . non-harness write to the RPC path
                          a process that outlives its tool call

   cache integrity        repository vs manifest, hourly
                          modified | unexpected | MISSING

   credential exposure    repos, CI logs, image layers, public internet
                          -> automatic revocation, minutes not hours

   exemption drift        live control state vs the approved register
```

D1.4 wrote detections for what an agent does to your estate. These are
detections for what happens to **the platform running it** — and they are
different rules, on different data, with different owners.

Four of them come straight out of the incident register, and each catches
something that looks like normal operation from inside the workload:

**Escape detection (C1.4).** Named primitives, not anomaly scoring: `ptrace`
attach to a non-child, `LD_PRELOAD` on a spawned process, writes to
`/proc/self/mem`, unexpected `unshare`/`setns`, a new listening socket, and —
the one the incident's spoofing depended on — a background process that
outlives the tool call that spawned it. Plus any process that is not the
harness writing to the harness's RPC path.

**Cache integrity diffing (C5.4).** An hourly comparison of what is in the
artifact repository against a manifest of what should be. Poisoned targets were
staged in exactly this way, and a deletion is as much a signal as a
modification.

**Secret scanning wired to revocation (C4.1).** Not only source: CI logs, image
layers, wikis, ticket attachments, and external exposure monitoring — the
incident's credentials were found on the public internet. Revocation has to be
automatic, because the interval between discovery and redistribution was
minutes.

**Exemption-state reconciliation (C6.3).** Live control state against the
approved register, continuously. Any control disabled without a matching
unexpired exemption is a P1, and expiry re-enables automatically with no grace
period.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">detection</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">owner</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">data source</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">escape detection (C1.4)</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">platform security</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">host sensor</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">cache integrity (C5.4)</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">build / release</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">artifact repository</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">secret scanning (C4.1)</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">appsec</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">repos, CI, the public internet</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">exemption drift (C6.3)</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">governance</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">control registry</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">upload scanning (C3.4)</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">the ingest team</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">upload path</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Five detections, five different owners, and not one of them reads the workload's own logs. That is the property that makes them useful after A2.8: they survive the subject of the investigation having had write access to everything else.</div>

## Your turn

Pick the orphaned-process rule and write it for your own platform. It is one query, it has a low false-positive rate, and on most agent platforms nobody has ever run it.

---

**Next → [D1.10 · Fleet-level correlation: seeing a swarm](https://spbreed.github.io/cyber-commons/lessons/D1.10.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D1.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D1.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*